# 5. GPT-2 inference with metrics


In [1]:
# Dynamicly load evaluation metrics and the model
%run -i ../src/models/evaluation.py
%run -i ../src/models/detoxGPT2.py

In [2]:
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
detoxGPT = detoxGPT2("../models/gpt2-based")

# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [8]:
prompt = "What a stupid thing to say!"

In [9]:
# Pack the suggestions into a dataframe
df = pd.DataFrame(
    detoxGPT.get_detoxed_suggestions(prompt, device=DEVICE), columns=["suggestion"]
)

# Add empty column for each metric
metrics = ["wo", "cs", "bleu"]
df[metrics] = pd.DataFrame([[0] * len(metrics)], index=df.index, dtype=float)

# Generate toxicity report for each suggestion
toxicity_report = toxicity.toxicity_report(df["suggestion"])
toxicity_report_columns = toxicity_report.columns

for index, row in df.iterrows():
    df.loc[index, "wo"] = similarity.get_wo_score(prompt, row["suggestion"])
    df.loc[index, "cs"] = similarity.get_cosine_score(prompt, row["suggestion"])
    df.loc[index, "bleu"] = similarity.get_bleu_score(prompt, row["suggestion"])

# Concat with toxicity report
df = pd.concat([df, toxicity_report], axis=1)
df

c:\Users\danielpancake\Desktop\text-detox\.venv\Lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\danielpancake\Desktop\text-detox\.venv\Lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\danielpancake\Desktop\text-detox\.venv\Lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.


,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,you're mad to say!,0.111111,0.823638,2.805139e-01,0.392323,0.003087,0.009576,0.000166,0.031821,0.002657
1,What a silly thing to say.,0.500000,0.978261,7.104154e-01,0.258028,0.000466,0.012569,0.000188,0.028832,0.001495
2,"I mean, this is a bad thing to say.",0.250000,0.895324,4.402099e-01,0.022341,0.001993,0.009369,0.000469,0.013132,0.000696
3,That's crazy - you're mad...,0.000000,0.633856,2.372749e-78,0.821724,0.014933,0.064402,0.000458,0.282791,0.040356
4,you just said it was bad.,0.000000,0.753464,3.987826e-78,0.060611,0.000942,0.011518,0.002880,0.009639,0.002009
5,you're mad...,0.000000,0.616387,5.115152e-232,0.514825,0.011588,0.043010,0.001189,0.085842,0.008416
6,it's all right,0.000000,0.692433,2.853520e-155,0.052903,0.000120,0.011566,0.002615,0.008307,0.002896
7,you're mad...,0.000000,0.616387,5.115152e-232,0.514825,0.011588,0.043010,0.001189,0.085842,0.008416
8,It's madness...,0.000000,0.593824,6.765882e-232,0.037838,0.001227,0.005018,0.000238,0.006115,0.000733
9,it was a bad thing to say,0.444444,0.881198,5.822370e-01,0.038117,0.001910,0.007144,0.000731,0.012488,0.001872


In [10]:
# Calculate the score
metric_weights = {"wo": 0.1, "cs": 0.5, "bleu": 0.4}

# Toxicity report should be as low as possible
# Similarity metrics should be as high as possible (use weighted sum)
df["score"] = 1 / df[toxicity_report_columns].sum(axis=1)
df["score"] *= df[metrics].dot(pd.Series(metric_weights))

# Sort by score
df = df.sort_values(by=["score"], ascending=False)
df

,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate,score
2,"I mean, this is a bad thing to say.",0.250000,0.895324,4.402099e-01,0.022341,0.001993,0.009369,0.000469,0.013132,0.000696,13.515656
9,it was a bad thing to say,0.444444,0.881198,5.822370e-01,0.038117,0.001910,0.007144,0.000731,0.012488,0.001872,11.530719
8,It's madness...,0.000000,0.593824,6.765882e-232,0.037838,0.001227,0.005018,0.000238,0.006115,0.000733,5.802635
6,it's all right,0.000000,0.692433,2.853520e-155,0.052903,0.000120,0.011566,0.002615,0.008307,0.002896,4.415632
4,you just said it was bad.,0.000000,0.753464,3.987826e-78,0.060611,0.000942,0.011518,0.002880,0.009639,0.002009,4.300652
1,What a silly thing to say.,0.500000,0.978261,7.104154e-01,0.258028,0.000466,0.012569,0.000188,0.028832,0.001495,2.729951
0,you're mad to say!,0.111111,0.823638,2.805139e-01,0.392323,0.003087,0.009576,0.000166,0.031821,0.002657,1.217238
5,you're mad...,0.000000,0.616387,5.115152e-232,0.514825,0.011588,0.043010,0.001189,0.085842,0.008416,0.463539
7,you're mad...,0.000000,0.616387,5.115152e-232,0.514825,0.011588,0.043010,0.001189,0.085842,0.008416,0.463539
3,That's crazy - you're mad...,0.000000,0.633856,2.372749e-78,0.821724,0.014933,0.064402,0.000458,0.282791,0.040356,0.258788


In [11]:
# Print the suggestion with the highest score
suggestion = df.iloc[0]["suggestion"]
print(f"{prompt} -> {suggestion}")

What a stupid thing to say! -> I mean, this is a bad thing to say.
